# 00 - Local Setup for GraphRAG Toolkit

This notebook sets up the AWS resources needed for GraphRAG toolkit and configures the environment for local execution.

## Prerequisites

Make sure you have:
1. AWS CLI configured with appropriate permissions
2. Python packages installed: `boto3`, `python-dotenv`, `json`
3. Sufficient AWS permissions for:
   - Neptune Analytics (neptune-graph:*)
   - OpenSearch Serverless (aoss:*)
   - AWS Bedrock (bedrock:*)
   - IAM (iam:CreateRole, iam:PutRolePolicy, iam:DeleteRole, iam:DeleteRolePolicy)
   - STS (sts:GetCallerIdentity)

Note: For production use, you should restrict these permissions to specific resources and actions.

In [33]:
import boto3
import json
import time
import os
from pathlib import Path

# Configuration
APPLICATION_ID = "graphrag-local-2"
REGION = "us-east-1"  # Change this to your preferred region
PROVISIONED_MEMORY = "16"

# Initialize AWS clients
neptune = boto3.client("neptune-graph", region_name=REGION)
opensearch = boto3.client("opensearchserverless", region_name=REGION)
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Setting up GraphRAG toolkit in region: {REGION}")
print(f"Application ID: {APPLICATION_ID}")

Setting up GraphRAG toolkit in region: us-east-1
Application ID: graphrag-local-2


## Step 1: Create IAM Role and Policies

First, let's create an IAM role that will be used by our services to interact with each other.


In [34]:
print("Creating IAM role and policies...")

try:
    # Initialize IAM client
    iam = boto3.client('iam', region_name=REGION)
    
    # Create IAM role
    role_name = f"{APPLICATION_ID}-role"
    
    assume_role_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": [
                        "neptune-graph.amazonaws.com",
                        "aoss.amazonaws.com",
                        "bedrock.amazonaws.com"
                    ]
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }
    
    try:
        role_response = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(assume_role_policy),
            Description=f"Role for GraphRAG toolkit {APPLICATION_ID}"
        )
        print(f"✅ IAM role created: {role_name}")
    except iam.exceptions.EntityAlreadyExistsException:
        print(f"ℹ️ IAM role already exists: {role_name}")
        role_response = iam.get_role(RoleName=role_name)
    
    role_arn = role_response['Role']['Arn']
    
    # Create policy for Neptune Analytics
    neptune_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "neptune-graph:*"
                ],
                "Resource": f"arn:aws:neptune-graph:{REGION}:*:graph/{APPLICATION_ID}-*"
            }
        ]
    }
    
    # Create policy for OpenSearch Serverless
    opensearch_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "aoss:*"
                ],
                "Resource": f"arn:aws:aoss:{REGION}:*:collection/{APPLICATION_ID}-*"
            }
        ]
    }
    
    # Create policy for Bedrock
    bedrock_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel"
                ],
                "Resource": [
                    f"arn:aws:bedrock:{REGION}::foundation-model/anthropic.claude-3-haiku-20240307-v1",
                    f"arn:aws:bedrock:{REGION}::foundation-model/amazon.titan-embed-text-v2"
                ]
            }
        ]
    }
    
    # Attach policies to role
    for policy_name, policy_doc in [
        ("neptune", neptune_policy),
        ("opensearch", opensearch_policy),
        ("bedrock", bedrock_policy)
    ]:
        policy_name = f"{APPLICATION_ID}-{policy_name}-policy"
        try:
            iam.put_role_policy(
                RoleName=role_name,
                PolicyName=policy_name,
                PolicyDocument=json.dumps(policy_doc)
            )
            print(f"✅ Attached policy: {policy_name}")
        except Exception as e:
            print(f"⚠️ Error attaching policy {policy_name}: {e}")
            raise
    
    print(f"\n✅ IAM setup complete!")
    print(f"Role ARN: {role_arn}")
    
except Exception as e:
    print(f"❌ Error in IAM setup: {e}")
    raise


Creating IAM role and policies...
✅ IAM role created: graphrag-local-2-role
✅ Attached policy: graphrag-local-2-neptune-policy
✅ Attached policy: graphrag-local-2-opensearch-policy
✅ Attached policy: graphrag-local-2-bedrock-policy

✅ IAM setup complete!
Role ARN: arn:aws:iam::533267284022:role/graphrag-local-2-role


## Step 1: Create Neptune Analytics Graph

In [35]:
print("Creating Neptune Analytics Graph...")


# Create Neptune Graph
graph_response = neptune.create_graph(
    graphName=f"{APPLICATION_ID}-graph",
    provisionedMemory=int(PROVISIONED_MEMORY),
    deletionProtection=False,
    publicConnectivity=True,
    replicaCount=1,
    vectorSearchConfiguration={
        "dimension": 1024
    }
)
 

Creating Neptune Analytics Graph...


In [36]:
graph_response

{'ResponseMetadata': {'RequestId': 'c94ece6e-bad8-42e7-bb6f-2b4819187e55',
  'HTTPStatusCode': 201,
  'HTTPHeaders': {'date': 'Mon, 07 Jul 2025 04:08:14 GMT',
   'content-type': 'application/json',
   'content-length': '411',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'c94ece6e-bad8-42e7-bb6f-2b4819187e55',
   'x-amz-apigw-id': 'NUg3IF6CoAMEAug=',
   'x-amzn-trace-id': 'Root=1-686b482d-273ecf843231cf417962ed71'},
  'RetryAttempts': 0},
 'id': 'g-4o5cjg3ix7',
 'name': 'graphrag-local-2-graph',
 'arn': 'arn:aws:neptune-graph:us-east-1:533267284022:graph/g-4o5cjg3ix7',
 'status': 'CREATING',
 'createTime': datetime.datetime(2025, 7, 6, 21, 8, 14, tzinfo=tzlocal()),
 'provisionedMemory': 16,
 'endpoint': 'g-4o5cjg3ix7.us-east-1.neptune-graph.amazonaws.com',
 'publicConnectivity': True,
 'vectorSearchConfiguration': {'dimension': 1024},
 'replicaCount': 1,
 'kmsKeyIdentifier': 'AWS_OWNED_KEY',
 'deletionProtection': False}

In [37]:
# Extract ID and endpoint from the response
graph_id = graph_response["id"]  # The field is called "id", not "graphId"
graph_endpoint = graph_response["endpoint"]

print(f"✅ Neptune Graph created successfully!")
print(f"   Graph ID: {graph_id}")
print(f"   Endpoint: {graph_endpoint}")
print(f"   Status: {graph_response['status']}")

# Wait for graph to be available
print("Waiting for graph to be available...")
waiter = neptune.get_waiter("graph_available")
waiter.wait(graphIdentifier=graph_id)  # Use graphIdentifier, not graphId
print("✅ Graph is now available!")

✅ Neptune Graph created successfully!
   Graph ID: g-4o5cjg3ix7
   Endpoint: g-4o5cjg3ix7.us-east-1.neptune-graph.amazonaws.com
   Status: CREATING
Waiting for graph to be available...
✅ Graph is now available!


## Step 2: Create OpenSearch Serverless Collection

In [38]:
print("Creating OpenSearch security policies...")

try:
    # Create encryption policy FIRST (required before collection creation)
    encryption_policy = {
        "Rules": [
            {
                "Resource": [f"collection/{APPLICATION_ID}-collection"],
                "ResourceType": "collection"
            }
        ],
        "AWSOwnedKey": True
    }
    
    try:
        opensearch.create_security_policy(
            name=f"{APPLICATION_ID}-encryption",
            policy=json.dumps(encryption_policy),
            type="encryption"
        )
        print("✅ Encryption policy created")
    except opensearch.exceptions.ConflictException:
        print("ℹ️ Encryption policy already exists")
    
    # Create network policy (also required before collection creation)
    network_policy = [
        {
            "Rules": [
                {
                    "Resource": [f"collection/{APPLICATION_ID}-collection"],
                    "ResourceType": "dashboard"
                },
                {
                    "Resource": [f"collection/{APPLICATION_ID}-collection"],
                    "ResourceType": "collection"
                }
            ],
            "AllowFromPublic": True
        }
    ]
    
    try:
        opensearch.create_security_policy(
            name=f"{APPLICATION_ID}-network",
            policy=json.dumps(network_policy),
            type="network"
        )
        print("✅ Network policy created")
    except opensearch.exceptions.ConflictException:
        print("ℹ️ Network policy already exists")
    
    # Create access policy
    # Get current user ARN
    sts = boto3.client("sts")
    user_arn = sts.get_caller_identity()["Arn"]
    
    access_policy = [
        {
            "Rules": [
                {
                    "Resource": [f"collection/{APPLICATION_ID}-collection"],
                    "Permission": [
                        "aoss:CreateCollectionItems",
                        "aoss:DeleteCollectionItems",
                        "aoss:UpdateCollectionItems",
                        "aoss:DescribeCollectionItems"
                    ],
                    "ResourceType": "collection"
                },
                {
                    "Resource": [f"index/{APPLICATION_ID}-collection/*"],
                    "Permission": [
                        "aoss:CreateIndex",
                        "aoss:DeleteIndex",
                        "aoss:UpdateIndex",
                        "aoss:DescribeIndex",
                        "aoss:ReadDocument",
                        "aoss:WriteDocument"
                    ],
                    "ResourceType": "index"
                }
            ],
            "Principal": [user_arn]
        }
    ]
    
    try:
        opensearch.create_access_policy(
            name=f"{APPLICATION_ID}-access",
            policy=json.dumps(access_policy),
            type="data"
        )
        print("✅ Access policy created")
    except opensearch.exceptions.ConflictException:
        print("ℹ️ Access policy already exists")
    
    print("✅ All security policies are ready!")
    
except Exception as e:
    print(f"❌ Error creating security policies: {e}")
    raise

Creating OpenSearch security policies...
✅ Encryption policy created
✅ Network policy created
✅ Access policy created
✅ All security policies are ready!


In [40]:
print("Creating OpenSearch Serverless Collection...")

try:
    # Create collection (now that policies exist)
    try:
        collection_response = opensearch.create_collection(
            name=f"{APPLICATION_ID}-collection",
            type="VECTORSEARCH",
            standbyReplicas="DISABLED"
        )
        
        collection_id = collection_response["createCollectionDetail"]["id"]
        
        print(f"✅ OpenSearch Collection created successfully!")
        print(f"   Collection ID: {collection_id}")
        print(f"   Status: {collection_response['createCollectionDetail']['status']}")
        
        # The collectionEndpoint is not available immediately after creation
        # We need to wait for the collection to be active and then retrieve it
        
    except opensearch.exceptions.ConflictException:
        print("ℹ️ Collection already exists, retrieving details...")
        # Get existing collection details
        collection_response = opensearch.batch_get_collection(names=[f"{APPLICATION_ID}-collection"])
        collection_detail = collection_response["collectionDetails"][0]
        collection_id = collection_detail["id"]
        
        print(f"✅ Using existing OpenSearch Collection:")
        print(f"   Collection ID: {collection_id}")
        print(f"   Status: {collection_detail['status']}")
    
    # Wait for collection to be active and get the endpoint
    print("Waiting for collection to be active...")
    while True:
        status_response = opensearch.batch_get_collection(names=[f"{APPLICATION_ID}-collection"])
        collection_detail = status_response["collectionDetails"][0]
        status = collection_detail["status"]
        print(f"   Collection status: {status}")
        
        if status == "ACTIVE":
            # Now we can get the endpoint
            collection_endpoint = collection_detail["collectionEndpoint"]
            print(f"   Collection endpoint: {collection_endpoint}")
            break
        elif status in ["FAILED", "DELETED"]:
            raise Exception(f"Collection creation failed with status: {status}")
        
        time.sleep(10)
    
    print("✅ Collection is now active!")
    
except Exception as e:
    print(f"❌ Error creating OpenSearch Collection: {e}")
    raise

Creating OpenSearch Serverless Collection...
ℹ️ Collection already exists, retrieving details...
✅ Using existing OpenSearch Collection:
   Collection ID: vgoh4hviohrcmbum3617
   Status: ACTIVE
Waiting for collection to be active...
   Collection status: ACTIVE
   Collection endpoint: https://vgoh4hviohrcmbum3617.us-east-1.aoss.amazonaws.com
✅ Collection is now active!


## Step 4: Create .env file for local execution

In [41]:
print("Creating .env file for local execution...")

try:
    # Determine region prefix for model names
    region_prefix = REGION.split("-")[0] if "-" in REGION else REGION
    
    # Create .env content with your specified models
    env_content = f"""# AWS Configuration
AWS_REGION={REGION}

# Graph Store (Neptune Analytics)
GRAPH_STORE=neptune-graph://{graph_id}

# Vector Store (OpenSearch Serverless)
VECTOR_STORE=aoss://{collection_endpoint}

# Model Configuration - Using your specified models
EXTRACTION_MODEL={region_prefix}.anthropic.claude-3-haiku-20240307-v1:0
EMBEDDINGS_MODEL=amazon.titan-embed-text-v2:0
EMBEDDINGS_DIMENSIONS=1024
RESPONSE_MODEL={region_prefix}.anthropic.claude-3-haiku-20240307-v1:0
EVALUATION_MODEL={region_prefix}.anthropic.claude-3-haiku-20240307-v1:0

# Neptune Notebook Configuration
GRAPH_NOTEBOOK_AUTH_MODE=IAM
GRAPH_NOTEBOOK_SSL=True
GRAPH_NOTEBOOK_IAM_PROVIDER=ROLE
GRAPH_NOTEBOOK_PORT=8182
GRAPH_NOTEBOOK_SERVICE=neptune-graph
GRAPH_NOTEBOOK_HOST={graph_endpoint}

# Application ID
APPLICATION_ID={APPLICATION_ID}
"""
    
    # Write .env file to repository root (go up one level from notebooks directory)
    repo_root = Path.cwd().parent
    env_file_path = repo_root / ".env"
    
    with open(env_file_path, "w") as f:
        f.write(env_content)
    
    print(f"✅ .env file created at: {env_file_path}")
    print("\nEnvironment variables configured:")
    print(f"  - GRAPH_STORE: neptune-graph://{graph_id}")
    print(f"  - VECTOR_STORE: aoss://{collection_endpoint}")
    print(f"  - EXTRACTION_MODEL: {region_prefix}.anthropic.claude-3-haiku-20240307-v1:0")
    print(f"  - EMBEDDINGS_MODEL: amazon.titan-embed-text-v2:0")
    print(f"  - RESPONSE_MODEL: {region_prefix}.anthropic.claude-3-haiku-20240307-v1:0")
    
except Exception as e:
    print(f"❌ Error creating .env file: {e}")
    raise

Creating .env file for local execution...
✅ .env file created at: /Users/manojs/Documents/Code/graphrag-toolkit/examples/lexical-graph/.env

Environment variables configured:
  - GRAPH_STORE: neptune-graph://g-4o5cjg3ix7
  - VECTOR_STORE: aoss://https://vgoh4hviohrcmbum3617.us-east-1.aoss.amazonaws.com
  - EXTRACTION_MODEL: us.anthropic.claude-3-haiku-20240307-v1:0
  - EMBEDDINGS_MODEL: amazon.titan-embed-text-v2:0
  - RESPONSE_MODEL: us.anthropic.claude-3-haiku-20240307-v1:0


## Step 5: Create extracted directory

In [ ]:
print("Creating extracted directory...")

try:
    repo_root = Path(__file__).parent.parent
    extracted_dir = repo_root / "extracted"
    extracted_dir.mkdir(exist_ok=True)
    
    print(f"✅ Extracted directory created at: {extracted_dir}")
    
except Exception as e:
    print(f"❌ Error creating extracted directory: {e}")
    raise

## Step 6: Update the 02-Separate-Extract-and-Build notebook

In [ ]:
# print("Updating 02-Separate-Extract-and-Build notebook to use ../.env...")

# try:
#     notebook_path = Path(__file__).parent / "02-Separate-Extract-and-Build.ipynb"
    
#     # Read the notebook
#     with open(notebook_path, "r") as f:
#         notebook_content = f.read()
    
#     # Replace %dotenv with %dotenv ../.env
#     updated_content = notebook_content.replace(
#         "%dotenv",
#         "%dotenv ../.env"
#     )
    
#     # Write back the updated notebook
#     with open(notebook_path, "w") as f:
#         f.write(updated_content)
    
#     print(f"✅ Updated notebook: {notebook_path}")
#     print("   Changed %dotenv to %dotenv ../.env")
    
# except Exception as e:
#     print(f"❌ Error updating notebook: {e}")
#     raise

In [ ]:
# # Fixed Resource Cleanup
# print("Starting cleanup of all resources...")

# try:
#     # 1. Delete Neptune Analytics Graph
#     print("\nCleaning up Neptune Analytics resources...")
#     try:
#         # Use correct parameters for Neptune Analytics
#         neptune.delete_graph(
#             graphIdentifier=graph_id,  # Use graphIdentifier, not graphId
#             skipSnapshot=True  # Required parameter
#         )
#         print("✅ Neptune Graph deleted")
#     except Exception as e:
#         print(f"⚠️ Error deleting Neptune Graph: {e}")

#     # 2. Delete OpenSearch resources
#     print("\nCleaning up OpenSearch resources...")
#     try:
#         # Get collection ID first
#         collection_name = f"{APPLICATION_ID}-collection"
#         try:
#             collection_response = opensearch.batch_get_collection(names=[collection_name])
#             collection_id = collection_response["collectionDetails"][0]["id"]
            
#             # Delete collection using ID
#             opensearch.delete_collection(id=collection_id)  # Use id, not name
#             print("✅ OpenSearch Collection deleted")
            
#         except Exception as e:
#             print(f"⚠️ Error getting/deleting collection: {e}")
        
#         # Delete security policies
#         for policy_type in ["network", "encryption"]:
#             try:
#                 opensearch.delete_security_policy(
#                     name=f"{APPLICATION_ID}-{policy_type}",
#                     type=policy_type
#                 )
#                 print(f"✅ {policy_type.title()} policy deleted")
#             except Exception as e:
#                 print(f"⚠️ Error deleting {policy_type} policy: {e}")
        
#         # Delete access policy
#         try:
#             opensearch.delete_access_policy(
#                 name=f"{APPLICATION_ID}-access",
#                 type="data"
#             )
#             print("✅ Access policy deleted")
#         except Exception as e:
#             print(f"⚠️ Error deleting access policy: {e}")
            
#     except Exception as e:
#         print(f"⚠️ Error cleaning up OpenSearch resources: {e}")

#     # 3. Delete IAM Role and Policies
#     print("\nCleaning up IAM resources...")
#     try:
#         role_name = f"{APPLICATION_ID}-role"
        
#         # Delete attached policies
#         for policy_name in ["neptune", "opensearch", "bedrock"]:
#             try:
#                 iam.delete_role_policy(
#                     RoleName=role_name,
#                     PolicyName=f"{APPLICATION_ID}-{policy_name}-policy"
#                 )
#                 print(f"✅ Deleted policy: {policy_name}")
#             except Exception as e:
#                 print(f"⚠️ Error deleting policy {policy_name}: {e}")
        
#         # Delete role
#         try:
#             iam.delete_role(RoleName=role_name)
#             print(f"✅ Deleted IAM role: {role_name}")
#         except Exception as e:
#             print(f"⚠️ Error deleting IAM role: {e}")
            
#     except Exception as e:
#         print(f"⚠️ Error cleaning up IAM resources: {e}")

#     # 4. Delete local files (fixed for Jupyter)
#     print("\nCleaning up local files...")
#     try:
#         # Use Path.cwd() instead of __file__ for Jupyter notebooks
#         repo_root = Path.cwd().parent
#         if repo_root.name == "notebooks":
#             repo_root = repo_root.parent
        
#         # Delete .env file
#         env_file = repo_root / ".env"
#         if env_file.exists():
#             env_file.unlink()
#             print("✅ Deleted .env file")
        
#         # Delete extracted directory
#         extracted_dir = repo_root / "extracted"
#         if extracted_dir.exists():
#             import shutil
#             shutil.rmtree(extracted_dir)
#             print("✅ Deleted extracted directory")
            
#         # Delete checkpoint files
#         checkpoint_files = ["extraction-checkpoint", "build-checkpoint"]
#         for checkpoint in checkpoint_files:
#             checkpoint_path = repo_root / checkpoint
#             if checkpoint_path.exists():
#                 shutil.rmtree(checkpoint_path)
#                 print(f"✅ Deleted {checkpoint}")
            
#     except Exception as e:
#         print(f"⚠️ Error cleaning up local files: {e}")

#     print("\n✅ Cleanup complete!")
#     print("💡 Please verify in the AWS Console that all resources have been properly deleted.")
    
# except Exception as e:
#     print(f"\n❌ Error during cleanup: {e}")
#     print("Some resources may still exist. Please check the AWS Console to ensure all resources are properly deleted.")

Starting cleanup of all resources...

Cleaning up Neptune Analytics resources...
✅ Neptune Graph deleted

Cleaning up OpenSearch resources...
✅ OpenSearch Collection deleted
✅ Network policy deleted
✅ Encryption policy deleted
✅ Access policy deleted

Cleaning up IAM resources...
⚠️ Error deleting policy neptune: An error occurred (NoSuchEntity) when calling the DeleteRolePolicy operation: The role with name graphrag-local-role cannot be found.
⚠️ Error deleting policy opensearch: An error occurred (NoSuchEntity) when calling the DeleteRolePolicy operation: The role with name graphrag-local-role cannot be found.
⚠️ Error deleting policy bedrock: An error occurred (NoSuchEntity) when calling the DeleteRolePolicy operation: The role with name graphrag-local-role cannot be found.
⚠️ Error deleting IAM role: An error occurred (NoSuchEntity) when calling the DeleteRole operation: The role with name graphrag-local-role cannot be found.

Cleaning up local files...
✅ Deleted .env file
✅ Delete

## Step 7: Test Bedrock access

In [ ]:
# print("Testing Bedrock access...")

# try:
#     # Test Titan embedding model
#     test_prompt = "Hello, world!"
    
#     # Test Titan embed model
#     titan_response = bedrock.invoke_model(
#         modelId="amazon.titan-embed-text-v2:0",
#         body=json.dumps({"inputText": test_prompt})
#     )
    
#     print("✅ Titan embedding model access confirmed")
    
#     # Test Claude Haiku model
#     region_prefix = REGION.split("-")[0] if "-" in REGION else REGION
#     claude_model = f"{region_prefix}.anthropic.claude-3-haiku-20240307-v1:0"
    
#     claude_response = bedrock.invoke_model(
#         modelId=claude_model,
#         body=json.dumps({
#             "messages": [{"role": "user", "content": test_prompt}],
#             "max_tokens": 100
#         })
#     )
    
#     print("✅ Claude Haiku model access confirmed")
    
# except Exception as e:
#     print(f"❌ Error testing Bedrock access: {e}")
#     print("Make sure you have the necessary Bedrock permissions and the models are available in your region.")
#     raise

## Setup Complete! 🎉

Your local GraphRAG environment is now ready. You can now run the  notebook from your local repository.

### Summary of what was created:
1. ✅ Neptune Analytics Graph
2. ✅ OpenSearch Serverless Collection
3. ✅ Security policies for OpenSearch
4. ✅  file with your specified models
5. ✅  directory
6. ✅ Updated notebook to use 
7. ✅ Tested Bedrock model access

### Next steps:
1. Run the  notebook
2. The notebook will now use:
   -  for embeddings
   -  for extraction, response, and evaluation
   - Your local environment instead of SageMaker

### To clean up resources later:
